In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features.feature_engineer.apm_features import _detect_star_players


In [3]:
def _opponent_season_stats_and_rank(df: pd.DataFrame) -> pd.DataFrame:
    """
    Opponent (team) features: YTD means through prior games only, plus weekly league
    rank among teams (1 = best) using each team's S2D stat at the last game in the
    ISO week, broadcast to all games in that week.
    """
    if 'OPP_OPP_ABBREVIATION_base' not in df.columns or 'SEASON_YEAR' not in df.columns:
        return df

    stat_cols = ['TEAM_DEF_RATING', 'TEAM_PACE', 'TEAM_POSS']
    available = [c for c in stat_cols if c in df.columns]
    if not available:
        return df

    need = ['GAME_ID', 'GAME_DATE', 'SEASON_YEAR', 'TEAM_ABBREVIATION'] + available
    team_game = (
        df[need]
        .drop_duplicates(subset=['GAME_ID', 'TEAM_ABBREVIATION'])
        .copy()
    )
    team_game['GAME_DATE'] = pd.to_datetime(team_game['GAME_DATE'], utc=False)
    team_game = team_game.sort_values(['SEASON_YEAR', 'TEAM_ABBREVIATION', 'GAME_DATE'])

    g = team_game.groupby(['SEASON_YEAR', 'TEAM_ABBREVIATION'], sort=False)

    season_renames = {}
    for col in available:
        base = col.replace('TEAM_', '')
        sname = f'{base}_season_to_date'
        team_game[sname] = g[col].transform(
            lambda x: x.expanding().mean().round(2)
        )
        season_renames[col] = sname

    team_game['RANK_WEEK'] = team_game['GAME_DATE'].dt.to_period('W-SAT')
    last_idx = team_game.groupby(
        ['SEASON_YEAR', 'RANK_WEEK', 'TEAM_ABBREVIATION'], sort=False
    )['GAME_DATE'].idxmax()
    weekly = team_game.loc[last_idx].copy()

    # Def rating: lower is better. Pace / poss: higher = rank 1.
    rank_asc = {
        'TEAM_DEF_RATING': True,
        'TEAM_PACE': False,
        'TEAM_POSS': False,
    }
    rank_cols = []
    for col, sname in season_renames.items():
        asc = rank_asc.get(col, True)
        rk = sname.replace('_season_to_date', '_szn_league_rank')
        rank_cols.append(rk)
        weekly[rk] = (
            weekly.groupby(['SEASON_YEAR', 'RANK_WEEK'])[sname]
            .rank(ascending=asc, method='min')
        )

    w = weekly[['SEASON_YEAR', 'RANK_WEEK', 'TEAM_ABBREVIATION', *rank_cols]]
    team_game = team_game.merge(
        w, on=['SEASON_YEAR', 'RANK_WEEK', 'TEAM_ABBREVIATION'], how='left'
    )

    out_cols = list(season_renames.values()) + rank_cols
    opp_rename = {c: f'OPP_{c}' for c in out_cols}
    team_game_opp = (
        team_game[['GAME_ID', 'TEAM_ABBREVIATION', *out_cols]]
        .rename(columns={**{'TEAM_ABBREVIATION': 'OPP_OPP_ABBREVIATION_base'}, **opp_rename})
    )

    return df.merge(team_game_opp, on=['GAME_ID', 'OPP_OPP_ABBREVIATION_base'], how='left')

In [4]:
# pd.set_option('display.max_columns', None)

# s21 = pd.read_csv('data/raw/season_stats/S21.csv').sort_values(by='GAME_DATE')
# p21 = pd.read_csv('data/raw/playoff_stats/P21.csv').sort_values(by='GAME_DATE')
# s21 = pd.concat([s21, p21])

# s22 = pd.read_csv('data/raw/season_stats/S22.csv').sort_values(by='GAME_DATE')
# p22 = pd.read_csv('data/raw/playoff_stats/P22.csv').sort_values(by='GAME_DATE')
# s22 = pd.concat([s22, p22])

# s23 = pd.read_csv('data/raw/season_stats/S23.csv').sort_values(by='GAME_DATE')
# p23 = pd.read_csv('data/raw/playoff_stats/P23.csv').sort_values(by='GAME_DATE')
# s23 = pd.concat([s23, p23])

s24 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S24.csv').sort_values(by='GAME_DATE')
s24 = _detect_star_players(s24)
s24 = _opponent_season_stats_and_rank(s24)
p24 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P24.csv').sort_values(by='GAME_DATE')
# s24 = pd.concat([s24, p24])

s25 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s25 = _detect_star_players(s25)
s25 = _opponent_season_stats_and_rank(s25)
p25 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
# s25 = pd.concat([s25, p25])

s26 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
s26 = _detect_star_players(s26)
s26 = _opponent_season_stats_and_rank(s26)
p26 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
p26 = _detect_star_players(p26)
p26 = _opponent_season_stats_and_rank(p26)
s26.sample(10)

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,...,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,OPP_DEF_RATING_season_to_date,OPP_PACE_season_to_date,OPP_POSS_season_to_date,OPP_DEF_RATING_szn_league_rank,OPP_PACE_szn_league_rank,OPP_POSS_szn_league_rank
22589,4056,2025-26,1641757,Jordan Miller,Jordan,1610612746,LAC,LA Clippers,22501012,2026-03-19T00:00:00,...,0,0,0,0,117.06,101.21,101.93,24.0,11.0,10.0
5901,20747,2025-26,1630191,Isaiah Stewart II,Isaiah,1610612765,DET,Detroit Pistons,22500060,2025-11-26T00:00:00,...,0,0,2,1,114.15,96.50,96.50,17.0,30.0,30.0
16281,10387,2025-26,1628997,Caleb Martin,Caleb,1610612742,DAL,Dallas Mavericks,22500721,2026-02-03T00:00:00,...,0,0,1,0,112.90,95.92,96.14,9.0,30.0,30.0
20282,6371,2025-26,1641708,Amen Thompson,Amen,1610612745,HOU,Houston Rockets,22500902,2026-03-05T00:00:00,...,1,0,3,1,112.86,100.49,100.95,11.0,15.0,17.0
21953,4695,2025-26,1629006,Josh Okogie,Josh,1610612745,HOU,Houston Rockets,22500989,2026-03-16T00:00:00,...,0,0,2,1,115.75,99.30,99.54,20.0,21.0,22.0
5175,21492,2025-26,203110,Draymond Green,Draymond,1610612744,GSW,Golden State Warriors,22500056,2025-11-21T00:00:00,...,0,0,2,1,116.98,104.03,104.56,21.0,3.0,5.0
456,26191,2025-26,1631108,Max Christie,Max,1610612742,DAL,Dallas Mavericks,22500096,2025-10-24T00:00:00,...,0,0,2,1,111.70,107.75,108.00,13.0,3.0,5.0
26224,428,2025-26,1630228,Jonathan Kuminga,Jonathan,1610612737,ATL,Atlanta Hawks,22501173,2026-04-10T00:00:00,...,0,0,2,1,114.15,100.70,101.05,15.0,13.0,15.0
6426,20219,2025-26,1630545,Terrence Shannon Jr.,Terrence,1610612750,MIN,Minnesota Timberwolves,22500306,2025-11-30T00:00:00,...,0,0,2,1,113.54,99.87,100.74,14.0,21.0,21.0
12378,14271,2025-26,1628997,Caleb Martin,Caleb,1610612742,DAL,Dallas Mavericks,22500546,2026-01-10T00:00:00,...,0,0,1,0,116.92,102.94,103.66,23.0,3.0,4.0


In [5]:
df = pd.concat([s26, p26])
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df['DAYS_REST'] = (
    df.groupby('PLAYER_ID')['GAME_DATE'].diff().dt.days.fillna(3))
df['IS_B2B']  = (df['DAYS_REST'] == 1).astype(int)
df['IS_HOME'] = df['MATCHUP'].str.contains('vs', na=False).astype(int)
df['TEAM_RATING_SMOOTH'] = df.groupby('TEAM_ID')['TEAM_NET_RATING'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean())
df['OPP_RATING_SMOOTH'] = df.groupby('OPP_TEAM_ID')['OPP_NET_RATING'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean())
df['EXPECTED_SPREAD'] = np.where(
    df['IS_HOME'] == 1,
    ((df['TEAM_RATING_SMOOTH'] - df['OPP_RATING_SMOOTH']) * -1.0) - 3.0,
    ((df['TEAM_RATING_SMOOTH'] - df['OPP_RATING_SMOOTH']) * -1.0) + 3.0)
df['EXPECTED_SPREAD'] = df['EXPECTED_SPREAD'].clip(lower=-20, upper=20)
df['BLOWOUT_RISK'] = (df['EXPECTED_SPREAD'].abs() >= 10).astype(int)
df.sample(10)

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,...,OPP_PACE_szn_league_rank,OPP_POSS_szn_league_rank,name,DAYS_REST,IS_B2B,IS_HOME,TEAM_RATING_SMOOTH,OPP_RATING_SMOOTH,EXPECTED_SPREAD,BLOWOUT_RISK
11327,15246,2025-26,1642859,Jase Richardson,Jase,1610612753,ORL,Orlando Magic,22500495,2026-01-04,...,10.0,13.0,NaN,2.0,0,1,-3.37,-8.95,-8.58,0
17433,9217,2025-26,1629723,John Konchar,John,1610612762,UTA,Utah Jazz,22500786,2026-02-11,...,18.0,19.0,NaN,2.0,0,1,6.07,-26.32,-20.00,1
25234,1439,2025-26,1630536,Sharife Cooper,Sharife,1610612764,WAS,Washington Wizards,22501133,2026-04-05,...,27.0,28.0,NaN,1.0,1,0,-15.10,-29.72,-11.62,1
10976,15673,2025-26,1629026,Kenrich Williams,Kenrich,1610612760,OKC,Oklahoma City Thunder,22500484,2026-01-02,...,18.0,20.0,NaN,2.0,0,0,28.94,2.65,-20.00,1
5027,21623,2025-26,1642066,Myron Gardner,Myron,1610612748,MIA,Miami Heat,22500051,2025-11-21,...,2.0,2.0,NaN,24.0,0,0,15.39,-0.72,-13.11,1
4144,22507,2025-26,1628970,Miles Bridges,Miles,1610612766,CHA,Charlotte Hornets,22500043,2025-11-14,...,14.0,15.0,NaN,2.0,0,0,-15.20,15.20,20.00,1
14618,12023,2025-26,1642857,Kasparas Jakučionis,Kasparas,1610612748,MIA,Miami Heat,22500649,2026-01-24,...,3.0,2.0,NaN,2.0,0,0,-8.42,-18.30,-6.88,0
246,26401,2025-26,201935,James Harden,James,1610612746,LAC,LA Clippers,22500087,2025-10-22,...,29.0,28.0,NaN,3.0,0,0,-22.10,22.10,20.00,1
24534,2116,2025-26,1642345,Oso Ighodaro,Oso,1610612756,PHX,Phoenix Suns,22501099,2026-03-31,...,14.0,14.0,NaN,1.0,1,0,4.67,-11.39,-13.06,1
18268,8378,2025-26,1642345,Oso Ighodaro,Oso,1610612756,PHX,Phoenix Suns,22500811,2026-02-21,...,16.0,14.0,NaN,2.0,0,1,-6.36,7.88,11.24,1


In [11]:
pdf = df[df['PLAYER_NAME'] == 'Chet Holmgren']
okc_stats = pdf[pdf['OPP_TEAM_ID'] == 1610612760]['PTS'].mean()
home_stats = pdf[pdf['IS_HOME'] == 1]
away_stats = pdf[pdf['IS_HOME'] == 0]
b2b_stats = pdf[pdf['IS_B2B'] == 1]
not_b2b_stats = pdf[pdf['IS_B2B'] == 0]
res_days_2_stats = pdf[pdf['DAYS_REST'] == 2]
res_days_3_stats = pdf[pdf['DAYS_REST'] == 3]
# active_star_count_1 = pdf[pdf['ACTIVE_STARS_COUNT'] == 1]
# active_star_count_2 = pdf[pdf['ACTIVE_STARS_COUNT'] == 2]
# active_star_count_3 = pdf[pdf['ACTIVE_STARS_COUNT'] == 3]
star_out_flag_1 = pdf[pdf['TOP_STAR_ACTIVE'] == 1]
star_out_flag_0 = pdf[pdf['TOP_STAR_ACTIVE'] == 0]
first_20_games = pdf['PTS'].iloc[:20].mean()
last_20_games = pdf['PTS'].iloc[-20:].mean()
last_3_games = pdf['PTS'].iloc[-3:].mean()
blowout_risk_1 = pdf[pdf['BLOWOUT_RISK'] == 1]
blowout_risk_0 = pdf[pdf['BLOWOUT_RISK'] == 0]
top_opponent_def_rating = pdf[pdf['OPP_DEF_RATING_szn_league_rank'] <= 5]
middle_opponent_def_rating = pdf[(pdf['OPP_DEF_RATING_szn_league_rank'] > 5) & (pdf['OPP_DEF_RATING_szn_league_rank'] <= 15)]
bottom_opponent_def_rating = pdf[pdf['OPP_DEF_RATING_szn_league_rank'] > 15]
high_opponent_pace = pdf[pdf['OPP_PACE_szn_league_rank'] <= 5]
middle_opponent_pace = pdf[(pdf['OPP_PACE_szn_league_rank'] > 5) & (pdf['OPP_PACE_szn_league_rank'] <= 15)]
low_opponent_pace = pdf[pdf['OPP_PACE_szn_league_rank'] > 15]


print(f"Main Player: {pdf['PLAYER_NAME'].iloc[0]} avg pts: {pdf['PTS'].mean().round(2)}")
print(f"First 20 games: {first_20_games.round(2)}")
print(f"Last 20 games: {last_20_games.round(2)}")
print(f"Last 3 games: {last_3_games.round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Home: {home_stats['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Away: {away_stats['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} B2B: {b2b_stats['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Not B2B: {not_b2b_stats['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Rest 2 Days: {res_days_2_stats['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Rest 3 Days: {res_days_3_stats['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Star Out Flag 1: {round(star_out_flag_1['PTS'].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Star Out Flag 0: {round(star_out_flag_0['PTS'].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Blowout Risk: {round(blowout_risk_1['PTS'].mean(), 2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Not Blowout Risk: {blowout_risk_0['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Top Opponent Def Rating: {top_opponent_def_rating['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Middle Opponent Def Rating: {middle_opponent_def_rating['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Bottom Opponent Def Rating: {bottom_opponent_def_rating['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} High Opponent Pace: {high_opponent_pace['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Middle Opponent Pace: {middle_opponent_pace['PTS'].mean().round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Low Opponent Pace: {low_opponent_pace['PTS'].mean().round(2)}")



Main Player: Chet Holmgren avg pts: 17.03
First 20 games: 18.55
Last 20 games: 16.6
Last 3 games: 15.0
Chet Holmgren Home: 17.14
Chet Holmgren Away: 16.92
Chet Holmgren B2B: 18.0
Chet Holmgren Not B2B: 16.89
Chet Holmgren Rest 2 Days: 16.97
Chet Holmgren Rest 3 Days: 17.81
Chet Holmgren Star Out Flag 1: 17.43
Chet Holmgren Star Out Flag 0: 15.0
Chet Holmgren Blowout Risk: 16.78
Chet Holmgren Not Blowout Risk: 18.07
Chet Holmgren Top Opponent Def Rating: 12.25
Chet Holmgren Middle Opponent Def Rating: 17.93
Chet Holmgren Bottom Opponent Def Rating: 17.41
Chet Holmgren High Opponent Pace: 17.27
Chet Holmgren Middle Opponent Pace: 17.6
Chet Holmgren Low Opponent Pace: 16.68


## HOME vs AWAY player avgs

In [43]:
# Mean minutes per player over all games in s26
min_by_player = df.groupby("PLAYER_ID")["MIN"].mean()
games_by_player = df.groupby("PLAYER_ID")["GAME_ID"].nunique()

eligible_ids = min_by_player.index[
    (min_by_player > 15) & (games_by_player.reindex(min_by_player.index) >= 20)
]

df = df[df["PLAYER_ID"].isin(eligible_ids)].copy()
df["IS_HOME"] = df["MATCHUP"].str.contains("vs", na=False)
pt_home_away = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_HOME"], as_index=False)
    .agg(PTS_PER_GAME=("PTS", "mean"), GAMES=("GAME_ID", "count"))
)

# Wide form: one row per player, columns for home / away
wide = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_HOME"])["PTS"]
    .mean()
    .unstack()
    .rename(columns={False: "away_pts", True: "home_pts"})
)
wide["home_minus_away"] = wide["home_pts"] - wide["away_pts"]
wide = wide.sort_values("home_minus_away", ascending=False)
wide.sample(10)

,IS_HOME,away_pts,home_pts,home_minus_away
PLAYER_ID,PLAYER_NAME,,,
1630536,Sharife Cooper,8.285714,7.950000,-0.335714
1628971,Bruce Brown,9.561798,8.554455,-1.007342
1628365,Markelle Fultz,5.676471,5.914286,0.237815
1626167,Myles Turner,14.454545,15.354545,0.900000
1631200,Kris Murray,4.631579,5.956989,1.325410
1630610,DeJon Jarreau,5.000000,7.818182,2.818182
203081,Damian Lillard,23.606557,25.457143,1.850585
1629216,Gabe Vincent,5.402985,5.260870,-0.142116
203915,Spencer Dinwiddie,11.105263,10.417722,-0.687542


## How players perform on Back 2 Back games

In [46]:
# After filtering to eligible_ids
df = df[df["PLAYER_ID"].isin(eligible_ids)].copy()

# Optional: if IS_B2B is int, this keeps 0/1 unstack labels predictable
# df["IS_B2B"] = df["IS_B2B"].astype(int)

b2b_summary = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_B2B"], as_index=False)
    .agg(PTS_PER_GAME=("PTS", "mean"), GAMES=("GAME_ID", "count"))
)

# Wide: one row per player — columns 0 and 1 if IS_B2B is int; False/True if bool
wide = (
    df.groupby(["PLAYER_ID", "PLAYER_NAME", "IS_B2B"])["PTS"]
    .mean()
    .unstack()
    .rename(columns={0: "pts_not_b2b", 1: "pts_b2b"})  # if bool: use False/True as keys
)

# How much better on rest vs B2B (positive = better with rest than on B2B)
wide["not_b2b_minus_b2b"] = wide["pts_not_b2b"] - wide["pts_b2b"]
wide = wide.sort_values("not_b2b_minus_b2b", ascending=False)
wide.sample(10)

,IS_B2B,pts_not_b2b,pts_b2b,not_b2b_minus_b2b
PLAYER_ID,PLAYER_NAME,,,
1627824,Guerschon Yabusele,8.500000,7.380952,1.119048
201976,Patrick Beverley,6.131148,6.416667,-0.285519
1641726,Dereck Lively II,8.500000,8.071429,0.428571
1630591,Jalen Suggs,13.468531,15.416667,-1.948135
1631106,Tari Eason,11.459677,7.133333,4.326344
203957,Dante Exum,7.803030,9.666667,-1.863636
1629627,Zion Williamson,22.750000,20.222222,2.527778
1629684,Grant Williams,9.018018,11.823529,-2.805511
1630573,Sam Hauser,8.907216,9.058824,-0.151607
